In [48]:
import pandas as pd
import numpy as np
from numpy.random import Generator, PCG64
from joblib import Parallel, delayed
import altair as alt

rng = Generator(PCG64())

In [63]:
all_df = pd.read_csv("./dataframes/all_evaluated_responses_df.csv")
all_df = all_df.dropna(subset=['agent_response'])
all_df['item_id'] = all_df['question'] + all_df['image_file']

In [82]:
item_all_df = all_df.groupby(['test_type', 'item_id', 'agent_type'])['is_correct'].mean().reset_index() #.sort_values(by='is_correct')
item_all_df.groupby(['test_type','agent_type'])['is_correct'].mean().reset_index()

,test_type,agent_type,is_correct
0,calvi,GPT-4V,0.353663
1,calvi,Human,0.495636
2,calvi,Salesforce/blip2-flan-t5-xl,0.226984
3,calvi,Salesforce/blip2-flan-t5-xxl,0.336891
4,calvi,google/matcha-chartqa,0.321280
5,calvi,google/pix2struct-chartqa-base,0.323214
6,calvi,liuhaotian/llava-v1.6-34b,0.261905
7,calvi,llava-hf/llava-1.5-13b-hf,0.282649
8,calvi,llava-hf/llava-1.5-7b-hf,0.293617
9,chartqa-test-continuous-human,GPT-4V,0.762832


In [105]:
temp = item_all_df[
    (item_all_df['test_type'] == 'holf2') &
    (item_all_df['agent_type'].isin(['Human', 'google/pix2struct-chartqa-base', 'google/matcha-chartqa']))
].pivot(index='item_id', columns='agent_type', values='is_correct').dropna()

temp[temp['google/pix2struct-chartqa-base'] != 0]

agent_type,Human,google/matcha-chartqa,google/pix2struct-chartqa-base
item_id,,,
"Imagine that you came across a random west coast state, which had a population of 1.75M people. What would you predict its number of crash fatalities to be? line_fatal_5_v2popc_v3region_v4age_v5years.png",0.0,0.0,0.20
What is the average reading score of students taught by suburban school teachers with 10-15 years of experience? bar_star_3_v2teacherc_v3area_v4none_v5none.png,1.0,0.0,1.00
What is the average reading score of students taught by suburban school teachers with 10-15 years of experience? bar_star_4_v2teacherc_v3area_v4math_v5none.png,1.0,0.0,0.10
What is the average reading score of students taught by suburban school teachers with 10-15 years of experience? bar_star_5_v2teacherc_v3area_v4math_v5birthyear.png,1.0,0.0,0.50
What is the average reading score of students taught by suburban school teachers with 10-15 years of experience? line_star_3_v2teacherc_v3area_v4none_v5none.png,1.0,0.0,0.70
What is the average reading score of students taught by suburban school teachers with 10-15 years of experience? line_star_4_v2teacherc_v3area_v4math_v5none.png,1.0,0.0,0.10
What is the average reading score of students taught by suburban school teachers with 10-15 years of experience? line_star_5_v2teacherc_v3area_v4math_v5birthyear.png,1.0,0.0,0.25


In [65]:
model_configs = [
    {'model': "Salesforce/blip2-flan-t5-xl", 'top_p': 'p06', 'temp': 't1', 'color': '#5ba3cf', 'accent_color': '#8cbedd'},
    {'model': "Salesforce/blip2-flan-t5-xxl", 'top_p': 'p1', 'temp': 't10', 'color': '#4c78a8', 'accent_color': '#81a0c2'},
    {'model': "llava-hf/llava-1.5-7b-hf", 'top_p': 'p04', 'temp': 't1', 'color': '#f9b574', 'accent_color': '#facb9d'},
    {'model': "llava-hf/llava-1.5-13b-hf", 'top_p': 'p1', 'temp': 't04', 'color': '#f58518', 'accent_color': '#f79d46'},
    {'model': "liuhaotian/llava-v1.6-34b", 'top_p': 'p1', 'temp': 't04', 'color': '#BF9000', 'accent_color': '#F1C232'},
    {'model': "google/pix2struct-chartqa-base", 'top_p': 'p08', 'temp': 't1', 'color': '#ba94c4', 'accent_color': '#c7b8d9'},
    {'model': "google/matcha-chartqa", 'top_p': 'p04', 'temp': 't1', 'color': '#8b6db2', 'accent_color': '#a28ac1'},
    {'model': "GPT-4V", 'top_p': 'p1', 'temp': 't02', 'color': '#c93739', 'accent_color': '#C6765E'},
    {'model': "Human", 'top_p': 'pna', 'temp': 'tna', 'color': '#2e693b', 'accent_color': '#73D287'},
]

### Find Human-Model Correlation

In [67]:
def bootstrap_correlation(
    model_responses, 
    human_responses,
    n_iterations=1000,
):
    """Added bootstrap coorelation """

    def bootstrap_iteration(
        model_responses,
        human_responses,
        chosen_items
    ):
        model_filter_df = pd.concat(
            [model_responses[model_responses['item_id'] == item] for item in chosen_items]
        )

        huamn_filter_df = pd.concat(
            [human_responses[human_responses['item_id'] == item] for item in chosen_items]
        )

        item_level_model_responses = model_filter_df.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        item_level_human_responses = huamn_filter_df.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        if item_level_model_responses['is_correct'].sum() == 0:
            human_model_correlation = np.nan
        else:
            human_model_correlation = np.corrcoef(
                item_level_model_responses['is_correct'],
                item_level_human_responses['is_correct']
            )[1][0]
            
        return human_model_correlation

    items = model_responses['item_id'].unique()
    bootstrapped_correlations = Parallel(n_jobs=-1)(
        delayed(bootstrap_iteration)(
            model_responses,
            human_responses,
            rng.choice(items, len(items), replace=True)
        ) for _ in range(n_iterations)
    )

    # get rid of all NA values
    bootstrapped_correlations = [bc for bc in bootstrapped_correlations if not pd.isna(bc)]

    if len(bootstrapped_correlations) == 0:
        return [np.nan, np.nan]
        
    lower = np.percentile(bootstrapped_correlations, 2.5)
    upper = np.percentile(bootstrapped_correlations, 97.5)

    return [lower, upper]

def human_model_correlation(all_df):
    """
    Find correlation on model and human responses on 
    """
    
    # grouped_df = all_df.groupby(['item_id', 'agent_type', 'test_type'])['is_correct'].mean().reset_index()
    human_model_corr_df = []

    for test in all_df['test_type'].unique():
        df = all_df[all_df['test_type'] == test].copy()
        print(test)
        
        human_responses = df[df['agent_type'] == 'Human']
        for agent in all_df['agent_type'].unique():
            if agent == 'Human':
                continue
            else:
                model_responses = df[df['agent_type'] == agent].copy()
                human_responses_valid = human_responses[
                    human_responses['item_id'].isin(model_responses['item_id'].unique())
                ]
            
            item_level_model_responses = model_responses.groupby(
                ['item_id', 'test_type']
            )['is_correct'].mean().reset_index().sort_values(by='item_id')

            item_level_human_responses = human_responses_valid.groupby(
                ['item_id', 'test_type']
            )['is_correct'].mean().reset_index().sort_values(by='item_id')

            # if the model only has incorrect responses for all questions
            if item_level_model_responses['is_correct'].sum() == 0:
                human_model_correlation = np.nan
            else:
                human_model_correlation = np.corrcoef(
                    item_level_model_responses['is_correct'],
                    item_level_human_responses['is_correct']
                )[1][0]

            lower, upper = bootstrap_correlation(
                model_responses,
                human_responses_valid
            )

            human_model_corr_df.append({
                'model': agent,
                'test': test,
                'lower_correlation': lower,
                'upper_correlation': upper,
                'correlation': human_model_correlation
            })

    return pd.DataFrame(human_model_corr_df)

try:
    print("reading human model correlation")
    human_model_correlation_df = pd.read_csv("./dataframes/human_model_correlation_df.csv")
except:
    print("general human model correlation")
    human_model_correlation_df = human_model_correlation(all_df)
    human_model_correlation_df.to_csv("./dataframes/human_model_correlation_df.csv")

general human model correlation
ggr
vlat
calvi
holf
holf2
chartqa-test-continuous-human


In [53]:
# human_model_correlation_df = human_model_correlation(all_df)

In [54]:
# human_model_correlation_df

### Find split-half reliability

In [68]:
def bootstrap_pairwise_correlation(
    human_responses,
    n_iterations=1000,
):
    """Split-half pairwise correlation bootstrapped over human responses"""

    def bootstrap_iteration(
        human_responses,
        chosen_items
    ):
        huamn_filter_df = pd.concat(
            [human_responses[human_responses['item_id'] == item] for item in chosen_items]
        ).sample(frac=1)

        human_half1 = huamn_filter_df.iloc[0:len(huamn_filter_df)//2]
        human_half2 = huamn_filter_df.iloc[len(huamn_filter_df)//2:]

        human_half1 = human_half1[human_half1['item_id'].isin(human_half2['item_id'])]
        human_half2 = human_half2[human_half2['item_id'].isin(human_half1['item_id'])]

        item_level_human_half1_responses = human_half1.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        item_level_human_half2_responses = human_half2.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        human_model_correlation = np.corrcoef(
            item_level_human_half1_responses['is_correct'],
            item_level_human_half2_responses['is_correct']
        )[1][0]
            
        return human_model_correlation

    items = human_responses['item_id'].unique()
    bootstrapped_correlations = Parallel(n_jobs=-1)(
        delayed(bootstrap_iteration)(
            human_responses,
            rng.choice(items, len(items), replace=True)
        ) for _ in range(n_iterations)
    )

    bootstrapped_correlations = [bc for bc in bootstrapped_correlations if not pd.isna(bc)]

    if len(bootstrapped_correlations) == 0:
        return [np.nan, np.nan]
        
    lower = np.percentile(bootstrapped_correlations, 2.5)
    upper = np.percentile(bootstrapped_correlations, 97.5)
    mean = np.mean(bootstrapped_correlations)

    return [mean, lower, upper]

def human_human_correlation(all_df):
    """
    Find correlation on model and human responses on 

    """
    
    human_model_corr_df = []

    for test in all_df['test_type'].unique():
        df = all_df[all_df['test_type'] == test].copy()
        
        human_responses = df[df['agent_type'] == 'Human']
        
        mean, lower, upper = bootstrap_pairwise_correlation(
            human_responses
        )
        human_model_corr_df.append({
            'model': 'Human',
            'test': test,
            'lower_correlation': lower,
            'upper_correlation': upper,
            'correlation': mean
        })

    return pd.DataFrame(human_model_corr_df)

try:
    print("reading human model correlation")
    human_human_correlation_df = pd.read_csv("./dataframes/human_human_correlation_df.csv")
except:
    print("general human model correlation")
    human_human_correlation_df = human_human_correlation(all_df)
    human_human_correlation_df.to_csv("./dataframes/human_human_correlation_df.csv")

general human model correlation


In [56]:
# human_human_correlation_df

### Correlation Error Bars

In [78]:
def create_human_machine_correlation_figure(correlation_df, tests, domain_order, domain_color=[]):
    """ Generate chart for point estimate and confidence intervals overlayed with item level scatteplots
    """
    charts = []
    for test in tests:
        df = correlation_df[correlation_df['test'] == test].copy()

        if test != 'ggr':
            axis =  alt.Axis(labels=False)
        else:
            axis = alt.Axis()
        
        mean_point_plot = alt.Chart(df).mark_point(filled=True, size=75, opacity=1).encode(
            x=alt.X('model:N', title='Category', scale=alt.Scale(domain=domain_order)),
            y=alt.Y('correlation:Q', title='', scale=alt.Scale(domain=[-1, 1]), axis=axis),
            color=alt.Color('model:N', scale=alt.Scale(domain=domain_order, range=domain_color), legend=None),
        )
        
        error_bars = alt.Chart(df).mark_rule(strokeWidth=2).encode(
            x=alt.X('model:N'),
            y=alt.Y('lower_correlation:Q'),
            y2='upper_correlation:Q',
            color=alt.Color('model:N', scale=alt.Scale(domain=domain_order, range=domain_color), legend=None)
        )

        split_half_band = alt.Chart(df[df['model'] == 'Human']).mark_rect(color=model_configs[-1]['color'], opacity=0.2).encode(
            y=alt.Y(f'lower_correlation:Q'),
            y2=(f'upper_correlation:Q'),
        )
    
        split_half_mean = alt.Chart(df[df['model'] == 'Human']).mark_rule(color=model_configs[-1]['color'], strokeDash=[5, 5]).encode(
            y=alt.Y(f'correlation:Q'),
            strokeWidth=alt.value(2)
        )

        if len(test) > 7:
            test = test[:7]
        
        chart = (mean_point_plot + error_bars + split_half_band + split_half_mean).properties(height=200, width=150, title=test)

        charts.append(chart)


    return alt.hconcat(*charts) #.resolve_scale(color='independent')

tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']
domain_order = [mc['model'] for mc in model_configs if mc['model'] != 'Human']
domain_color = [mc['color'] for mc in model_configs if mc['model'] != 'Human']

correlation_df = pd.concat([
    human_model_correlation_df,
    human_human_correlation_df
])

human_machine_correlation_figure = create_human_machine_correlation_figure(
    correlation_df,
    tests=tests,
    domain_order=domain_order, 
    domain_color=domain_color,
)
human_machine_correlation_figure.save("./figures/human_machine_correlation.pdf")
human_machine_correlation_figure

alt.HConcatChart(...)

In [70]:
# correlation_df[correlation_df['test'] == 'holf2'].sort_values(by=['correlation'])

In [71]:
correlation_df

,model,test,lower_correlation,upper_correlation,correlation
0,Salesforce/blip2-flan-t5-xl,ggr,0.063909,0.343654,0.151787
1,Salesforce/blip2-flan-t5-xxl,ggr,0.038428,0.317256,0.151787
2,llava-hf/llava-1.5-7b-hf,ggr,NaN,NaN,NaN
3,llava-hf/llava-1.5-13b-hf,ggr,0.024071,0.328005,0.151787
4,liuhaotian/llava-v1.6-34b,ggr,-0.857409,0.255369,0.063363
5,google/pix2struct-chartqa-base,ggr,NaN,NaN,NaN
6,google/matcha-chartqa,ggr,NaN,NaN,NaN
7,GPT-4V,ggr,0.019966,0.454784,0.209490
8,Salesforce/blip2-flan-t5-xl,vlat,0.052660,0.397153,0.232002
9,Salesforce/blip2-flan-t5-xxl,vlat,-0.040923,0.280163,0.121689


## Difference of means

In [83]:
def bootstrap_diff_of_mean_ci(
        data: pd.DataFrame,
        measure,
        id_col,
        group1: str,
        group2: str,
        group_col='agent_type',
        n_iterations=1000,
        statistic=np.mean,
    ):
    
    items = list(data[id_col].unique())
    n_size = len(items)
    df = data.copy()

    def bootstrap_iteration(data, chosen_items):
        filter_df = pd.concat(
            [data[data[id_col].isin([item])] for item in chosen_items]
        )
        
        bs_mean = statistic(filter_df[measure]) 
        return (bs_mean, list(chosen_items))

    
    qset_means1 = Parallel(n_jobs=-1)(
        delayed(bootstrap_iteration)(
            df[df[group_col] == group1].copy(),
            rng.choice(items, n_size,  replace=True)
        ) for _ in range(n_iterations)
    )

    qset_means2 = Parallel(n_jobs=-1)(
        delayed(bootstrap_iteration)(
            df[df[group_col] == group2].copy(),
            rng.choice(items, n_size,  replace=True)
        ) for _ in range(n_iterations)
    )
    
    means = []
    qs_used = []
    means1 = [bs_mean for bs_mean, chosen_qs in qset_means1]
    means2 = [bs_mean for bs_mean, chosen_qs in qset_means2]
    means = np.subtract(means1, means2)
 
    # 95% confidence interval
    lower = np.percentile(means, 2.5)
    upper = np.percentile(means, 97.5)
    
    return lower, upper

def create_diff_of_mean_ci_df(
    data: pd.DataFrame,
    measure,
    id_col,
    group1,
    group2,
    group_col='agent_type',
    statistic=np.mean
):
    """ create the dataframe for 95% bootstrapped confidence interval
    data: dataset
    measure: dependent variable
    id_col: units to bootstrap along (e.g. participant_d or item_id)
    condition_col: the different conditions of the experiments
    """
    
    data_list = []
    lower, upper = bootstrap_diff_of_mean_ci(
        data,
        group1=group1,
        group2=group2,
        measure=measure, 
        statistic=statistic,
        group_col=group_col,
        id_col=id_col)
    
    mean1 = data[data[group_col] == group1][measure].mean()
    mean2 = data[data[group_col] == group2][measure].mean()
    mean = mean1 - mean2
    
    data_list.append({
        "mean": mean,
        "ci_upper": upper, 
        "ci_lower": lower,
    })

    return pd.DataFrame(data_list)

## Caclulate interactions 

In [84]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [85]:
%%R
install.packages("lme4")
library(lme4)


The downloaded binary packages are in
	/var/folders/v8/3zpbxkws53b3x6m8509jyml80000gn/T//Rtmp8zddF4/downloaded_packages


trying URL 'https://cran.case.edu/bin/macosx/big-sur-arm64/contrib/4.4/lme4_1.1-36.tgz'
Content type 'application/x-gzip' length 7079300 bytes (6.8 MB)
downloaded 6.8 MB



## Gap between humand and models

In [86]:
# %%R -i human_model_correlation_df

# base_model <- lmer(
#     correlation ~ 1 + model + test , 
#     data = human_model_correlation_df,
# )

# interaction_model <- lmer(
#     correlation ~ 1 + model * test, 
#     data = human_model_correlation_df, 
# )

# anova(base_model, interaction_model, test="LRT")

In [ ]:
gpt_human_performance_df = group_average_df[group_average_df['agent_type'].isin(['GPT-4V', 'Human'])]
gpt_human_performance_df = gpt_human_performance_df.groupby(
    ['agent_type', 'item_id', 'task_category', 'chart_type']
)['is_correct'].mean().reset_index()

In [44]:
human_model_correlation_df

,Unnamed: 0,model,test,lower_correlation,upper_correlation,correlation
0,0,Salesforce/blip2-flan-t5-xl,ggr,0.002089,0.306175,0.139554
1,1,Salesforce/blip2-flan-t5-xxl,ggr,0.008778,0.310854,0.139554
2,2,llava-hf/llava-1.5-7b-hf,ggr,NaN,NaN,NaN
3,3,llava-hf/llava-1.5-13b-hf,ggr,-0.010402,0.309260,0.139554
4,4,liuhaotian/llava-v1.6-34b,ggr,-0.920200,0.233886,0.038864
5,5,google/pix2struct-chartqa-base,ggr,NaN,NaN,NaN
6,6,google/matcha-chartqa,ggr,NaN,NaN,NaN
7,7,GPT-4V,ggr,-0.020076,0.393436,0.190099
8,8,Salesforce/blip2-flan-t5-xl,vlat,0.030413,0.433784,0.228250
9,9,Salesforce/blip2-flan-t5-xxl,vlat,-0.089610,0.269252,0.098217
